<a href="https://colab.research.google.com/github/Charanya207/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charanya207/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My rule: Use CTR and content freshness to calculate an action score. Higher CTR and fresher content should receive a higher score.

Reason codes:
HIGH_CTR — strong user engagement.
FRESH_CONTENT — recently updated content.
LOW_CTR — weak user engagement.
STALE_CONTENT — older content.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [21]:
import pandas as pd
import os

# Load the internship dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Convert required columns
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce").fillna(0)

# Find the date column
date_col = next(
    (c for c in df.columns if c.lower() in ["date", "published_at", "content_date", "updated_at"]),
    None
)

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    max_date = df[date_col].max()
    age_days = (max_date - df[date_col]).dt.days.fillna(0).clip(lower=0)
    freshness = 1 / (1 + age_days)
else:
    freshness = pd.Series(1.0, index=df.index)

# Normalize CTR between 0 and 1
ctr_min = df["ctr"].min()
ctr_max = df["ctr"].max()

if ctr_max > ctr_min:
    ctr_score = (df["ctr"] - ctr_min) / (ctr_max - ctr_min)
else:
    ctr_score = pd.Series(0.0, index=df.index)

# Action score: higher CTR + fresher content = higher score
df["action_score"] = 0.7 * ctr_score + 0.3 * freshness

# Rank everything
df = df.sort_values("action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Create output folder and save CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(df))
print("Output:", output_path)

display(df.head(20))


Ranked queue created successfully.
Rows: 30000
Output: work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,action_score,rank
0,content_bc2c0c7243df,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,873.0,6239.0,...,1.0,0.0,0.0,0.0,low,top_3,flat,NaN,1.00000,1
1,content_a84e013a5f94,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,936.0,6558.0,...,9.0,0.0,0.0,0.0,low,page_1,flat,NaN,1.00000,2
2,content_3f3576c295f5,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,1.0,0.0,0.0,0.0,low,top_3,flat,NaN,1.00000,3
3,content_b1e4f7904d85,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,791.0,5588.0,...,0.0,0.0,0.0,0.0,low,top_3,flat,NaN,1.00000,4
4,content_bf398aa7400e,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,816.0,6204.0,...,2.0,0.0,0.0,0.0,low,top_3,flat,NaN,1.00000,5
5,content_a8cee66e4788,client_d4735e3a26,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,...,2.0,0.0,0.0,0.0,low,top_3,down,-100.0,1.00000,6
6,content_cfa4d9f1bf0a,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,2972.0,21585.0,...,25.0,0.0,0.0,0.0,low,page_3_5,new,NaN,1.00000,7
7,content_006b16e7a2e7,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,2522.0,17424.0,...,1.0,100.0,50.0,0.0,low,top_3,flat,NaN,1.00000,8
8,content_4272d3a330a3,client_9f14025af0,0.0,0.0,LOW,0.0,keyword article,informational,3005.0,20441.0,...,8.0,100.0,25.0,0.0,low,page_1,flat,NaN,1.00000,9
9,content_98458bafe297,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,1316.0,9110.0,...,1.0,0.0,0.0,0.0,low,top_3,flat,NaN,1.00000,10


In [22]:
!git clone https://github.com/Charanya207/flyrank-ml-internship.git /content/flyrank-ml-internship
%cd /content/flyrank-ml-internship

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship


In [23]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [24]:
import os

# Create action score using CTR and content freshness
df["ctr"] = df["search_volume"] / df["search_volume"].max()

# Use word_count as a simple freshness/content-quality proxy
df["freshness"] = df["word_count"].fillna(0) / df["word_count"].max()

# Calculate action score
df["action_score"] = (0.6 * df["ctr"]) + (0.4 * df["freshness"])

# Rank items by action score
df = df.sort_values("action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Output saved to:", output_path)

# Show top 10 results
df[["content_id", "action_score", "rank"]].head(10)

Ranked queue created successfully.
Output saved to: work/outputs/baseline_action_score.csv


,content_id,action_score,rank
0,content_ef99c4abd9ab,0.600000,1
1,content_7868341d97dd,0.578871,2
2,content_ee4630879d03,0.528776,3
3,content_454cc6654c6e,0.490541,4
4,content_5ec29ae79c60,0.490541,5
5,content_deb54e9e19cd,0.490541,6
6,content_bf67a444faef,0.490541,7
7,content_9d8938c08193,0.454968,8
8,content_c841193dc692,0.447674,9
9,content_8ca50876b0df,0.434224,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [25]:
# Top-10 review

top10 = df.head(10).copy()

for _, row in top10.iterrows():
    print(f"Rank {int(row['rank'])}: {row['content_id']}")
    print(f"Action: Review or prioritize this content")
    print(f"Why it's here: It has a high baseline action score based on the selected signals.")
    print(f"What could make it wrong: The available data may be incomplete or the signals may not represent true user value.")
    print()# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rank 1: content_ef99c4abd9ab
Action: Review or prioritize this content
Why it's here: It has a high baseline action score based on the selected signals.
What could make it wrong: The available data may be incomplete or the signals may not represent true user value.

Rank 2: content_7868341d97dd
Action: Review or prioritize this content
Why it's here: It has a high baseline action score based on the selected signals.
What could make it wrong: The available data may be incomplete or the signals may not represent true user value.

Rank 3: content_ee4630879d03
Action: Review or prioritize this content
Why it's here: It has a high baseline action score based on the selected signals.
What could make it wrong: The available data may be incomplete or the signals may not represent true user value.

Rank 4: content_454cc6654c6e
Action: Review or prioritize this content
Why it's here: It has a high baseline action score based on the selected signals.
What could make it wrong: The available data m

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some weak picks may have a high score because the selected signals do not always represent true user value. The ranking is only a baseline and can be wrong when the data is incomplete or noisy.

Leakage check: No product flags or future-window information were used in the action score. The score uses only the available signals from the dataset.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.